# Experiment Runner Demo

This notebook demonstrates how to use the config-driven experiment runner to run the full pipeline for a single meeting or all meetings.

## Prerequisites

- Ensure `.venv` is activated
- Cached ASR outputs (Whisper JSON) and embeddings (ECAPA .pt) should exist in `results/asr/` and `results/embeddings/`
- Annotations in `data/processed/`
- **Dataset cache**: The AMI dataset is stored on the external drive at `/media/ikkjo/StoreJet - Ilija/hf_cache`. The experiment configs already include `hf_cache_dir` pointing there.

## Commands (for terminal usage)

```bash
# Run a single experiment
.venv/bin/python experiments/run_experiment.py --config experiments/configs/scenario2_ihm.json

# Run with dry-run (validate only)
.venv/bin/python experiments/run_experiment.py --config experiments/configs/scenario2_ihm.json --dry-run

# Run on a single meeting
.venv/bin/python experiments/run_experiment.py --config experiments/configs/scenario2_ihm.json --meeting-id EN2001a

# Run all experiments
.venv/bin/python experiments/run_all_experiments.py

# Run only IHM experiments
.venv/bin/python experiments/run_all_experiments.py --mic ihm

# Run only scenario 3 experiments
.venv/bin/python experiments/run_all_experiments.py --scenario scenario3
```

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from experiments.run_experiment import load_config, validate_config, discover_meetings, run_experiment

config_path = project_root / "experiments/configs/scenario2_ihm.json"
config = load_config(config_path)
print(f"Experiment: {config['experiment_id']}")
print(f"Scenario: {config['scenario']}")
print(f"Mic: {config['microphone_configuration']}")
print(f"Seed: {config['seed']}")

## Validate config and discover meetings (dry-run)

In [ ]:
# Run dry-run to validate and discover meetings without running models
result = run_experiment(
    config_path=config_path,
    dry_run=True,
)

print(f"\nDry-run result:")
print(f"  Output dir: {result['output_dir']}")
print(f"  Discovered meetings: {result['summary'].get('num_meetings_discovered', 0)}")

## Run experiment on a single meeting

In [ ]:
# Run on a single meeting for quick testing
result = run_experiment(
    config_path=config_path,
    meeting_id="EN2001a",
)

print(f"\nExperiment completed: {result['experiment_id']}")
print(f"Output directory: {result['output_dir']}")
print(f"Meetings: {result['summary']['num_completed']}/{result['summary']['num_meetings_total']} completed")

if result['summary'].get('der', {}).get('mean'):
    print(f"Mean DER: {result['summary']['der']['mean']:.3f}")
if result['summary'].get('wer_integrated', {}).get('mean'):
    print(f"Mean WER: {result['summary']['wer_integrated']['mean']:.3f}")

## Run full experiment (all meetings)

In [ ]:
# Run on all meetings (this may take a while)
# Uncomment to run:

# result = run_experiment(
#     config_path=config_path,
# )

# print(f"\nExperiment completed: {result['experiment_id']}")
# print(f"Output directory: {result['output_dir']}")
# print(f"Meetings: {result['summary']['num_completed']}/{result['summary']['num_meetings_total']} completed")
# print(f"Mean DER: {result['summary']['der']['mean']:.3f}")
# print(f"Mean WER: {result['summary']['wer_integrated']['mean']:.3f}")

## Inspect output directory structure

In [ ]:
output_dir = Path(result['output_dir'])

print("Output structure:")
for item in sorted(output_dir.rglob("*")):
    if item.is_file():
        print(f"  {item.relative_to(output_dir)}")

## Load and inspect per-meeting metrics

In [ ]:
import json

metrics_dir = output_dir / "metrics"
for metrics_file in sorted(metrics_dir.glob("*.json")):
    if metrics_file.name == "summary.json":
        continue
    with open(metrics_file, "r") as f:
        metrics = json.load(f)
    print(f"\n{metrics['meeting_id']}:")
    print(f"  DER: {metrics['der']['der']:.3f}")
    print(f"  JER: {metrics['jer']['jer']:.3f}")
    print(f"  WER (integrated): {metrics['wer']['integrated']['wer']:.3f}")
    if metrics['wer'].get('whisper_only'):
        print(f"  WER (whisper only): {metrics['wer']['whisper_only']['wer']:.3f}")
    if metrics.get('speaker_identification'):
        print(f"  Speaker ID accuracy: {metrics['speaker_identification']['segment_accuracy']:.3f}")

## Load and inspect summary

In [ ]:
summary_path = output_dir / "metrics" / "summary.json"
if summary_path.exists():
    with open(summary_path, "r") as f:
        summary = json.load(f)
    
    print("Summary:")
    print(json.dumps(summary, indent=2))
else:
    print("Summary not found - run the experiment first")

## Run all experiments via Python API

In [ ]:
from experiments.run_all_experiments import run_all_experiments

# Uncomment to run all experiments (this will take a long time)

# all_results = run_all_experiments(
#     config_dir=project_root / "experiments/configs",
# )

# print(f"\nAll experiments complete!")
# print(f"Results saved to: {all_results['all_runs_path']}")
# print(f"Completed: {len([r for r in all_results['all_runs'] if r['status'] == 'completed'])}")
# print(f"Failed: {len([r for r in all_results['all_runs'] if r['status'] == 'failed'])}")